In [1]:
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import shap
import pickle
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X_train = pd.read_parquet("../data/processed/X_train.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")

X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_test = pd.read_parquet("../data/processed/y_test.parquet")

In [3]:
cols_woes = []
cols_norm = []
for col in X_train.columns:
    if "_woe" in col:
        cols_woes.append(col)
    else:
        cols_norm.append(col)

## Model LGBM

In [ ]:
skf = StratifiedKFold(    n_splits=6,    shuffle=True,   random_state=42 )
lgb_model_norm = lgb.LGBMClassifier(objective="binary",metric="auc")
lgb_model_woe = lgb.LGBMClassifier(objective="binary",metric="auc")

In [8]:
import optuna
import lightgbm as lgb
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score
)

In [9]:
def optimize_lgbm(X, y, n_trials=30):

    def objective(trial):

        params = {
            "objective": "binary",
            "metric": "auc",
            "random_state": 42,
            "verbosity": -1,

            "n_estimators": trial.suggest_int(
                "n_estimators",
                100,
                1000
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.1,
                log=True
            ),

            "num_leaves": trial.suggest_int(
                "num_leaves",
                15,
                63
            ),

            "min_child_samples": trial.suggest_int(
                "min_child_samples",
                20,
                100
            ),

            "subsample": trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            )
        }

        model = lgb.LGBMClassifier(**params)

        cv = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        )

        auc = cross_val_score(
            model,
            X,
            y,
            scoring="roc_auc",
            cv=cv,
            n_jobs=-1
        ).mean()

        return auc

    study = optuna.create_study(
        direction="maximize"
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True
    )

    return study.best_params

In [10]:
best_params_norm = optimize_lgbm(
    X_train[cols_norm],
    y_train,
    n_trials=30
)

lgb_norm = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    random_state=42,
    verbosity=-1,
    **best_params_norm
)

lgb_norm.fit(
    X_train[cols_norm],
    y_train
)

[I 2026-06-07 20:47:57,748] A new study created in memory with name: no-name-433156f3-116d-42c2-851f-c6956571d429
Best trial: 0. Best value: 0.86135:   3%|▎         | 1/30 [00:09<04:30,  9.34s/it]

[I 2026-06-07 20:48:07,090] Trial 0 finished with value: 0.8613495112701546 and parameters: {'n_estimators': 704, 'learning_rate': 0.02156413458622268, 'num_leaves': 45, 'min_child_samples': 62, 'subsample': 0.7219672426022923, 'colsample_bytree': 0.7234938825847111}. Best is trial 0 with value: 0.8613495112701546.


Best trial: 0. Best value: 0.86135:   7%|▋         | 2/30 [00:14<03:20,  7.17s/it]

[I 2026-06-07 20:48:12,739] Trial 1 finished with value: 0.8610209662346561 and parameters: {'n_estimators': 447, 'learning_rate': 0.05606786017298576, 'num_leaves': 23, 'min_child_samples': 53, 'subsample': 0.8823661803260492, 'colsample_bytree': 0.8082830782353121}. Best is trial 0 with value: 0.8613495112701546.


Best trial: 0. Best value: 0.86135:  10%|█         | 3/30 [00:21<03:11,  7.07s/it]

...

Best trial: 8. Best value: 0.865047:  97%|█████████▋| 29/30 [01:32<00:02,  2.28s/it]

[I 2026-06-07 20:49:30,215] Trial 28 finished with value: 0.8645419172131128 and parameters: {'n_estimators': 263, 'learning_rate': 0.018074550564253066, 'num_leaves': 17, 'min_child_samples': 39, 'subsample': 0.8053760807708207, 'colsample_bytree': 0.7346113621788003}. Best is trial 8 with value: 0.8650472430203555.


Best trial: 8. Best value: 0.865047: 100%|██████████| 30/30 [01:33<00:00,  3.12s/it]
d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


[I 2026-06-07 20:49:31,485] Trial 29 finished with value: 0.8649564537740714 and parameters: {'n_estimators': 163, 'learning_rate': 0.04197207285558186, 'num_leaves': 23, 'min_child_samples': 58, 'subsample': 0.7228981842157257, 'colsample_bytree': 0.7004499915365229}. Best is trial 8 with value: 0.8650472430203555.


,boosting_type,'gbdt'
,num_leaves,16
,max_depth,-1
,learning_rate,0.02205723238746582
,n_estimators,348
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,35


In [11]:
best_params_woe = optimize_lgbm(
    X_train[cols_woes],
    y_train,
    n_trials=30
)

lgb_woe = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    random_state=42,
    verbosity=-1,
    **best_params_woe
)

lgb_woe.fit(
    X_train[cols_woes],
    y_train
)

[I 2026-06-07 20:49:33,228] A new study created in memory with name: no-name-750925f9-4a61-428e-9994-54c03ef7b45c
Best trial: 0. Best value: 0.85828:   3%|▎         | 1/30 [00:02<01:16,  2.64s/it]

[I 2026-06-07 20:49:35,871] Trial 0 finished with value: 0.8582802949024652 and parameters: {'n_estimators': 570, 'learning_rate': 0.06040825362777144, 'num_leaves': 18, 'min_child_samples': 72, 'subsample': 0.933202387619066, 'colsample_bytree': 0.9075232219104339}. Best is trial 0 with value: 0.8582802949024652.


Best trial: 0. Best value: 0.85828:   7%|▋         | 2/30 [00:06<01:29,  3.19s/it]

[I 2026-06-07 20:49:39,449] Trial 1 finished with value: 0.8509913287452658 and parameters: {'n_estimators': 563, 'learning_rate': 0.06436990116933558, 'num_leaves': 46, 'min_child_samples': 21, 'subsample': 0.8180610809818774, 'colsample_bytree': 0.9543338448884606}. Best is trial 0 with value: 0.8582802949024652.


Best trial: 0. Best value: 0.85828:  10%|█         | 3/30 [00:10<01:37,  3.62s/it]

...

Best trial: 12. Best value: 0.863157:  97%|█████████▋| 29/30 [01:44<00:03,  3.49s/it]

[I 2026-06-07 20:51:17,773] Trial 28 finished with value: 0.8630938751945184 and parameters: {'n_estimators': 549, 'learning_rate': 0.011712742528581725, 'num_leaves': 20, 'min_child_samples': 100, 'subsample': 0.7967900290060035, 'colsample_bytree': 0.7590891160422136}. Best is trial 12 with value: 0.8631571866171015.


Best trial: 12. Best value: 0.863157: 100%|██████████| 30/30 [01:47<00:00,  3.60s/it]
d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


[I 2026-06-07 20:51:21,185] Trial 29 finished with value: 0.862277402669898 and parameters: {'n_estimators': 535, 'learning_rate': 0.01617901176542343, 'num_leaves': 34, 'min_child_samples': 100, 'subsample': 0.8313315314456751, 'colsample_bytree': 0.8531660133354837}. Best is trial 12 with value: 0.8631571866171015.


,boosting_type,'gbdt'
,num_leaves,15
,max_depth,-1
,learning_rate,0.017512554498848764
,n_estimators,744
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,67


In [12]:
def evaluate_model(
    model,
    X_test,
    y_test,
    model_name
):

    y_prob = model.predict_proba(X_test)[:, 1]

    y_pred = model.predict(X_test)

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    gini = 2 * auc - 1

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    return {
        "model": model_name,
        "auc": round(auc, 4),
        "gini": round(gini, 4),
        "accuracy": round(accuracy, 4)
    }

In [13]:
result_norm = evaluate_model(
    lgb_norm,
    X_test[cols_norm],
    y_test,
    "LGBM_NORM"
)

result_woe = evaluate_model(
    lgb_woe,
    X_test[cols_woes],
    y_test,
    "LGBM_WOE"
)

results = pd.DataFrame([
    result_norm,
    result_woe
])


In [15]:
results

,model,auc,gini,accuracy
0,LGBM_NORM,0.8665,0.7330,0.9372
1,LGBM_WOE,0.8637,0.7274,0.9368
